# Collega i tuoi schemi alle pagine del prof (solo codice, niente AI)

Carichi i tuoi schemi e le pagine del prof (PDF o PNG). Per ogni pagina di schema il codice trova la pagina del prof più simile:
- **testo** (PDF con testo): parole rare in comune, pesate IDF (`annotatore/prepara.py`);
- **vista** (PNG o scansioni senza testo): hash percettivo + istogramma dei grigi. Funziona male fra schema disegnato e slide: guarda il punteggio.

Poi, per le coppie, conta quante frasi del prof ritrovi nello schema e lancia `trova_differenze.py` con le parole delle pagine del prof collegate. I file restano in questa sessione Colab.

## 1. Scarica il repo e installa

In [ ]:
!git clone -q https://github.com/03gilbe-design/pacchetto-schemi-studio.git
%cd pacchetto-schemi-studio/strumenti/2_collega_pagine_prof
!pip install -q -r ../3_numeri_da_appunti/requirements.txt
!python -m playwright install --with-deps chromium > /dev/null 2>&1 && echo chromium pronto

## 2. Carica i file
Prima i tuoi schemi, poi le pagine del prof. Terzo upload facoltativo: pagine generate da un'AI (`.html/.svg/.png`) per `trova_differenze.py`; se annulli usa gli esempi del repo.

In [ ]:
from google.colab import files
import os, shutil
for d in ('miei', 'prof', 'generate'): os.makedirs('caricati/' + d, exist_ok=True)
def carica(d, msg):
    print(msg); fs = list(files.upload())
    for f in fs: shutil.move(f, f'caricati/{d}/{f}')
    return [f'caricati/{d}/{f}' for f in fs]
miei = carica('miei', '1) i tuoi schemi (PDF o PNG)')
prof = carica('prof', '2) le pagine del prof (PDF o PNG)')
gen = carica('generate', '3) facoltativo: pagine generate da un AI (annulla per usare gli esempi)')
GEN = 'caricati/generate' if gen else '../3_numeri_da_appunti/esempi_numeri/generate'

## 3. Collega e confronta

In [ ]:
a = ' '.join(f'"{f}"' for f in miei); b = ' '.join(f'"{f}"' for f in prof)
!python -X utf8 collega_deterministico.py --schemi {a} --prof {b} --generate "{GEN}" --out risultati 2>&1 | grep -v MuPDF
from IPython.display import Image, Markdown, display
display(Image('risultati/differenze/classifica.png'))
display(Markdown(open('risultati/differenze/FRASI_PER_IL_MODELLO.md', encoding='utf-8').read()))

## 4. Scarica i risultati

In [ ]:
!cd risultati && zip -qr ../collegamenti.zip .
files.download('collegamenti.zip')